# Study 898 — Managed-Vol Equity — the teardown

The HAC alpha regression, the leverage-timing decomposition, the 3×3 grid, the paired Sharpe-gap bootstrap, the 200-seed shuffled-signal placebo, the two-era cut, the cost sweep, and the 30-seed synthetic control. Excess-of-cash on both legs (SPY − BIL); one documented rebalance lag.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_rows': 4802, 'n_days': 4780, 'fingerprint': 'f7ef586be44c', 'sh_strat': 0.659, 'vol_strat': 13.4, 'dd_strat': -31.1, 'cagr_strat': 8.26, 'wealth_strat': 4.51, 'sh_bh': 0.549, 'vol_bh': 19.9, 'dd_bh': -56.5, 'cagr_bh': 9.35, 'wealth_bh': 5.45, 'sharpe_gap': 0.11, 'alpha': 2.78, 't_alpha': 1.67, 'beta': 0.56, 'appraisal': 0.36, 'avg_w': 0.96, 'share_lev': 41.5, 'turnover': 9.15, 'exposure_bps': 2.407, 'timing_bps': 1.102, 'diff_bps': -0.822, 't_diff': -0.97, 'boot_gap': 0.11, 'boot_lo': -0.128, 'boot_hi': 0.355, 'boot_pneg': 0.195, 'vol_median': 12.5, 'vol_p10': 8.8, 'vol_p90': 17.4, 'vol_band': 86.0, 'bh_vol_p90': 27.8, 'era_early_t': 0.96, 'era_early_sh': 0.449, 'era_early_shbh': 0.347, 'era_early_n': 2143, 'era_late_t': 1.08, 'era_late_sh': 0.827, 'era_late_shbh': 0.759, 'era_late_n': 2637, 'cost1_sh': 0.652, 'cost1_alpha': 2.69, 'cost5_sh': 0.613, 'cost5_alpha': 2.16, 'cost5_dd': -31.4, 'pl_alpha_obs': 2.78, 'pl_alpha_mean': -0.11, 'pl_alpha_sd': 2.03, 'pl_p_alpha': 0.07, 'pl_gap_obs': 0.11, 'pl_gap_mean': -0.055, 'pl_p_gap': 0.05, 'pl_dd_obs': -31.1, 'pl_dd_mean': -57.4, 'pl_p_dd': 0.0, 'syn_null_t': -0.01, 'syn_null_fire': 7, 'syn_planted_t': 4.98, 'syn_planted_alpha': 9.4, 'syn_planted_fire': 97, 'crash': [('GFC 2008-09', -30.6, -56.5), ('2018 Q4', -18.0, -19.8), ('COVID 2020', -13.7, -33.9), ('2022 bear', -15.2, -25.0)]}

## The headline — managed-vol vs buy-and-hold SPY (excess of cash)

In [2]:
print(f"managed : Sharpe {R['sh_strat']:.3f}  vol {R['vol_strat']:.1f}%  "
      f"maxDD {R['dd_strat']:.1f}%  excess-CAGR {R['cagr_strat']:.2f}%  x{R['wealth_strat']:.2f}")
print(f"buy&hold: Sharpe {R['sh_bh']:.3f}  vol {R['vol_bh']:.1f}%  "
      f"maxDD {R['dd_bh']:.1f}%  excess-CAGR {R['cagr_bh']:.2f}%  x{R['wealth_bh']:.2f}")
print(f"Sharpe gap {R['sharpe_gap']:+.3f} | HAC alpha {R['alpha']:+.2f}%/yr "
      f"(t={R['t_alpha']:+.2f}, beta={R['beta']:.2f}, appraisal={R['appraisal']:.2f})")
print(f"avg weight {R['avg_w']:.2f}, levered {R['share_lev']:.1f}% of days, "
      f"turnover {R['turnover']:.2f}x NAV/yr, n={R['n_days']}")

managed : Sharpe 0.659  vol 13.4%  maxDD -31.1%  excess-CAGR 8.26%  x4.51
buy&hold: Sharpe 0.549  vol 19.9%  maxDD -56.5%  excess-CAGR 9.35%  x5.45
Sharpe gap +0.110 | HAC alpha +2.78%/yr (t=+1.67, beta=0.56, appraisal=0.36)
avg weight 0.96, levered 41.5% of days, turnover 9.15x NAV/yr, n=4780


## Is it 'just' leverage-timing? The decomposition

A *constant* scale leaves the Sharpe unchanged, so the whole Sharpe gap **is** the timing term. Split `mean(managed) = β·mean(B&H) [exposure] + α [timing]`:

In [3]:
print(f"exposure (beta*mean B&H) {R['exposure_bps']:+.3f} bps/day")
print(f"timing   (alpha)         {R['timing_bps']:+.3f} bps/day")
print(f"net managed - B&H daily  {R['diff_bps']:+.3f} bps  (HAC t = {R['t_diff']:+.2f})")
print('beta<1: the book GIVES UP raw excess return; the timing alpha buys a lower-vol path.')

exposure (beta*mean B&H) +2.407 bps/day
timing   (alpha)         +1.102 bps/day
net managed - B&H daily  -0.822 bps  (HAC t = -0.97)
beta<1: the book GIVES UP raw excess return; the timing alpha buys a lower-vol path.


## Placebo — shuffle the vol signal (200 seeds; same weights, no timing)

In [4]:
print(f"HAC alpha : obs {R['pl_alpha_obs']:+.2f}% vs placebo {R['pl_alpha_mean']:+.2f}% "
      f"(sd {R['pl_alpha_sd']:.2f}) -> p = {R['pl_p_alpha']:.3f}")
print(f"Sharpe gap: obs {R['pl_gap_obs']:+.3f} vs placebo {R['pl_gap_mean']:+.3f} -> p = {R['pl_p_gap']:.3f}")
print(f"max DD    : obs {R['pl_dd_obs']:+.1f}% vs placebo {R['pl_dd_mean']:+.1f}% -> p = {R['pl_p_dd']:.3f}")
print('The tail shield is unambiguous timing (p=0.000); the alpha/Sharpe is only borderline.')

HAC alpha : obs +2.78% vs placebo -0.11% (sd 2.03) -> p = 0.070
Sharpe gap: obs +0.110 vs placebo -0.055 -> p = 0.050
max DD    : obs -31.1% vs placebo -57.4% -> p = 0.000
The tail shield is unambiguous timing (p=0.000); the alpha/Sharpe is only borderline.


## Bootstrap — the Sharpe advantage, paired circular block

In [5]:
print(f"gap {R['boot_gap']:+.3f}  95% CI [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]  "
      f"Pr[gap<0] = {R['boot_pneg']:.3f}  -> straddles zero")

gap +0.110  95% CI [-0.128, +0.355]  Pr[gap<0] = 0.195  -> straddles zero


## Robustness — two eras (split 2016-01-01)

In [6]:
print(f"2007-2016 (n={R['era_early_n']}): alpha t = {R['era_early_t']:+.2f}  "
      f"Sharpe {R['era_early_sh']:.3f} (bh {R['era_early_shbh']:.3f})")
print(f"2016-2026 (n={R['era_late_n']}): alpha t = {R['era_late_t']:+.2f}  "
      f"Sharpe {R['era_late_sh']:.3f} (bh {R['era_late_shbh']:.3f})")
print('Sign stable in both halves; significance never arrives.')

2007-2016 (n=2143): alpha t = +0.96  Sharpe 0.449 (bh 0.347)
2016-2026 (n=2637): alpha t = +1.08  Sharpe 0.827 (bh 0.759)
Sign stable in both halves; significance never arrives.


## The timer — costs (one-way bps x |dw| + borrow on the levered leg)

In [7]:
print(f"1 bp        : Sharpe {R['cost1_sh']:.3f}  alpha {R['cost1_alpha']:+.2f}%")
print(f"5 bp + 1%b  : Sharpe {R['cost5_sh']:.3f}  alpha {R['cost5_alpha']:+.2f}%  maxDD {R['cost5_dd']:.1f}%")
print('Cheap to run; what fails the bar gross also fails it net.')

1 bp        : Sharpe 0.652  alpha +2.69%
5 bp + 1%b  : Sharpe 0.613  alpha +2.16%  maxDD -31.4%
Cheap to run; what fails the bar gross also fails it net.


## Synthetic positive control — the machinery is unbiased (live)

Null (risk priced) must NOT fire; planted leverage-effect must light up.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from managed_vol import strategy as st
null_t = np.array([st.synthetic_detect(0.0, seed=898+s, n_days=4000)['t_alpha'] for s in range(8)])
plant_t = np.array([st.synthetic_detect(2.0, seed=898+s, n_days=4000)['t_alpha'] for s in range(8)])
print(f"null    (disconnect=0), 8 seeds: alpha t mean {null_t.mean():+.2f} "
      f"(sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
print(f"planted (disconnect=2), 8 seeds: alpha t mean {plant_t.mean():+.2f}, "
      f"t>=2 in {(plant_t>=2).sum()}/8")
print(f"(frozen 30-seed run: null mean t {R['syn_null_t']:+.2f} fires {R['syn_null_fire']}%; "
      f"planted mean t {R['syn_planted_t']:+.2f} fires {R['syn_planted_fire']}%)")

null    (disconnect=0), 8 seeds: alpha t mean +0.43 (sd 1.15), |t|>=2 in 1/8
planted (disconnect=2), 8 seeds: alpha t mean +4.75, t>=2 in 8/8
(frozen 30-seed run: null mean t -0.01 fires 7%; planted mean t +4.98 fires 97%)


## Verdict

- **Signal — MIXED (Real on the tail control · Weak on the Sharpe).** The drawdown-taming half is real on the real tape: full-sample max DD **-31.1% vs -56.5%**, robust across all nine grid cells and both eras, certified as genuine *timing* by the shuffled-signal placebo (**p = 0.000**, 200 seeds); with average weight 0.96 it de-risks in storms, not just holds less. The return half is **not** certified: HAC alpha **+2.78%/yr at t = 1.67** (era t = +0.96/+1.08), and the +0.110 Sharpe gain has a bootstrap CI [-0.128, +0.355] straddling zero. Single ~19-year SPY tape — short-history, named.
- **Tradability — FRAGILE.** Turnover 9.15× NAV/yr costs ~9 bps/yr at 1 bp, SPY capacity is unlimited, and every number survives 5 bp + 1% borrow (Sharpe 0.613). But what survives certification is **risk control, not excess return**: β = 0.56 < 1 means a ~1.1 pp/yr excess-CAGR give-up for the smoother path, and the Sharpe uplift never clears *t* = 2 — a real shield, not a bankable edge. FRAGILE, not INVESTABLE.